# Epithelial sub-cluster + morphology in napari

Trimmed copy of `05_napari_morphology.ipynb`: loads the lightweight `cell_id` + `sub_leiden` export from `05_subclustering_epithelial.ipynb` (instead of the global `cluster_labels.csv`), merges it onto the full-slide coordinates/boundaries, and opens it alongside the Xenium morphology image in napari. Non-epithelial cells (absent from the export) are shown in gray.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ubuntu + Wayland
os.environ["QT_QPA_PLATFORM"] = "xcb"
os.environ["PYOPENGL_PLATFORM"] = "glx"

import napari

In [ ]:
PROJECT = Path.home() / "Projects/xenium_lung"

VIS_DIR = PROJECT / "results/xenium_5k/visualization"
RAW_DIR = PROJECT / "data/bundle/xenium_prime_5k"

COORDINATES_FILE = VIS_DIR / "cell_coordinates.parquet"
BOUNDARIES_FILE = VIS_DIR / "cell_boundaries.parquet"

# lightweight per-compartment export from 05_subclustering_epithelial.ipynb (cell_id + sub_leiden + sub_cell_type)
CLUSTERS_FILE = PROJECT / "results/xenium_5k/annotation/subclustering/epithelial/sub_cluster_labels.csv"
COLOR_COL = "sub_leiden"  # switch to "sub_cell_type" once subcluster_labels has been filled in

MORPHOLOGY_FILE = RAW_DIR / "morphology.ome.tif"

print("Clusters:", CLUSTERS_FILE)
print("Exists:", CLUSTERS_FILE.exists())
print("Morphology:", MORPHOLOGY_FILE)
print("Exists:", MORPHOLOGY_FILE.exists())

# Load the cell information

In [ ]:
coordinates = pd.read_parquet(COORDINATES_FILE)
clusters = pd.read_csv(CLUSTERS_FILE)[["cell_id", COLOR_COL]].rename(columns={COLOR_COL: "group"})

cells = coordinates.merge(
    clusters,
    on="cell_id",
    how="left",
    validate="one_to_one"
)

print("Cells:", cells.shape)
print(cells.head())

In [ ]:
cluster_ids = sorted(
    cells["group"].dropna().unique()
)

cmap = plt.get_cmap(
    "tab20",
    len(cluster_ids)
)

cluster_colors = {
    cluster_id: cmap(i)
    for i, cluster_id in enumerate(cluster_ids)
}

point_colors = np.array([
    cluster_colors[group]
    if not pd.isna(group)
    else (0.5, 0.5, 0.5, 1.0)
    for group in cells["group"]
])

print("Clusters:", cluster_ids)

In [ ]:
import re
import tifffile

with tifffile.TiffFile(MORPHOLOGY_FILE) as tif:
    ome_xml = tif.ome_metadata

# cell x/y are in microns; the TIFF pixel grid needs this scale to line up with them
match_x = re.search(r'PhysicalSizeX="([\d.]+)"', ome_xml)
match_y = re.search(r'PhysicalSizeY="([\d.]+)"', ome_xml)

pixel_size_x = float(match_x.group(1)) if match_x else 1.0
pixel_size_y = float(match_y.group(1)) if match_y else 1.0

print("Pixel size (x, y):", pixel_size_x, pixel_size_y)

# Open napari

In [ ]:
viewer = napari.Viewer()

# Load morphology image

The morphology TIFF is a multiscale (pyramidal) OME-TIFF — reading the full series would require 64 GiB of RAM. Instead build a list of dask arrays, one per resolution level, so only visible tiles are read from disk.

In [ ]:
import dask.array as da
import zarr


def _open_level_array(level):
    node = zarr.open(level.aszarr(), mode="r")
    if isinstance(node, zarr.Array):
        return node
    # multi-array group — the dataset for this level is the first array member
    key = next(iter(node.array_keys()))
    return node[key]


with tifffile.TiffFile(MORPHOLOGY_FILE) as tif:
    series = tif.series[0]
    levels = series.levels

    pyramid = [
        da.from_array(_open_level_array(level), chunks="auto")
        for level in levels
    ]

for i, level in enumerate(pyramid):
    print(i, "shape =", level.shape, "dtype =", level.dtype)

In [ ]:
viewer.add_image(
    pyramid,
    name="Morphology",
    multiscale=True,
    contrast_limits=[0, 20000],
    scale=(pixel_size_y, pixel_size_x),
)

# Add cells

Colored by `COLOR_COL` (default `sub_leiden`); cells outside the Epithelial compartment are gray.

In [ ]:
all_points = cells[["y", "x"]].to_numpy(
    dtype=np.float32
)

viewer.add_points(
    all_points,
    size=2,
    face_color=point_colors,
    name=f"Cells — {COLOR_COL}",
)

# Add boundaries

Not adding all boundaries yet — reuse the boundary subset function from `05_napari_morphology.ipynb`.

In [ ]:
from shapely import from_wkb

def add_boundary_subset(
    viewer,
    boundaries,
    cluster_lookup,
    cluster_colors,
    n_polygons=5000,
    layer_name=None,
):

    subset = boundaries.iloc[:n_polygons]

    geometries = from_wkb(
        subset["geometry"].to_numpy()
    )

    polygon_data = []
    polygon_groups = []

    for cell_id, geom in zip(
        subset.index,
        geometries
    ):

        group = cluster_lookup.get(
            str(cell_id),
            np.nan
        )

        if geom.geom_type == "Polygon":

            coords = np.asarray(
                geom.exterior.coords
            )

            # Shapely X,Y → napari Y,X
            coords = coords[:, [1, 0]]

            polygon_data.append(coords)
            polygon_groups.append(group)

        elif geom.geom_type == "MultiPolygon":

            for polygon in geom.geoms:

                coords = np.asarray(
                    polygon.exterior.coords
                )

                coords = coords[:, [1, 0]]

                polygon_data.append(coords)
                polygon_groups.append(group)

    edge_colors = np.array([
        cluster_colors[group]
        if not pd.isna(group)
        else (0.5, 0.5, 0.5, 1.0)
        for group in polygon_groups
    ])

    if layer_name is None:
        layer_name = (
            f"Cell boundaries — {n_polygons:,}"
        )

    return viewer.add_shapes(
        polygon_data,
        shape_type="polygon",
        edge_width=0.8,
        edge_color=edge_colors,
        face_color=[0, 0, 0, 0],
        name=layer_name,
    )

In [ ]:
boundaries = pd.read_parquet(
    BOUNDARIES_FILE
)

cluster_lookup = cells.set_index(
    "cell_id"
)["group"]

boundary_layer = add_boundary_subset(
    viewer,
    boundaries,
    cluster_lookup,
    cluster_colors,
    n_polygons=5_000,
)